# Unit 2 — Advanced Visualization & Storytelling
## Global Development Monitor


In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
import shap
gapminder=px.data.gapminder(); centroids=pd.read_csv('country_centroids_computed.csv'); gap_latest=gapminder[gapminder.year==2007].merge(centroids,on='iso_alpha',how='left').assign(total_gdp=lambda d:d['pop']*d.gdpPercap)


In [ ]:
px.choropleth(gap_latest,locations='iso_alpha',color='total_gdp',hover_name='country',title='Total GDP by country').show(); px.choropleth(gap_latest,locations='iso_alpha',color='gdpPercap',hover_name='country',title='GDP per Capita by country').show(); print(gap_latest.nlargest(5,'total_gdp')[['country','total_gdp']]); print(gap_latest.nlargest(5,'gdpPercap')[['country','gdpPercap']])


In [ ]:
X_model=gapminder[['gdpPercap','pop','year']].values; y_model=gapminder.lifeExp.values; rf=RandomForestRegressor(n_estimators=200,random_state=0).fit(X_model,y_model); shap_values=shap.TreeExplainer(rf).shap_values(X_model); row_pos=gapminder.index.get_loc(gapminder.index[(gapminder.country=='Norway')&(gapminder.year==2007)][0]); vals=shap_values[row_pos]; plt.barh(['gdpPercap','pop','year'],vals,color=['#3b6ea5' if v>0 else '#b5432e' for v in vals]); plt.axvline(0,color='black'); plt.title('SHAP: Norway 2007'); plt.show()


In [ ]:
def bootstrap_ci(values,n_boot=2000,seed=0):
    rng=np.random.default_rng(seed); boot=np.array([rng.choice(values,len(values),replace=True).mean() for _ in range(n_boot)]); return values.mean(),np.percentile(boot,2.5),np.percentile(boot,97.5)
a,b='Africa','Europe'; ra=bootstrap_ci(gap_latest[gap_latest.continent==a].lifeExp.values); rb=bootstrap_ci(gap_latest[gap_latest.continent==b].lifeExp.values); fig,ax=plt.subplots();
for i,(name,res) in enumerate([(a,ra),(b,rb)]): mean,lo,hi=res; ax.errorbar(i,mean,yerr=[[mean-lo],[hi-mean]],fmt='o',capsize=6)
ax.set_xticks([0,1]); ax.set_xticklabels([a,b]); ax.set_ylabel('Mean life expectancy, 95% CI'); plt.show()


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.5)); sc=axes[0].scatter(gap_latest.centroid_lon,gap_latest.centroid_lat,c=gap_latest.lifeExp,cmap='viridis'); fig.colorbar(sc,ax=axes[0]); axes[0].set_title('Country centroids by life expectancy'); trend=gapminder.groupby('year').lifeExp.mean(); axes[1].plot(trend.index,trend.values,'o-',color='navy'); axes[1].set_title('World average life expectancy'); pop=gap_latest.groupby('continent').pop.sum().sort_values(); axes[2].barh(pop.index,pop.values,color='teal'); axes[2].set_title('Population by continent'); plt.tight_layout(); plt.show()
